# Track 9: Money laundering on a payment network

Starter notebook. Run it from top to bottom in a fresh Colab session. It
clones your repository, downloads the data, loads it in a memory-friendly
way, shows the basic numbers, and hands you the fixed test split. Then it
stops, and the `TODO D1.x` cells mark where your work begins.

**Data:** Elliptic, Kaggle `ellipticco/elliptic-data-set`: 203,769 Bitcoin transactions as nodes, 234,355 payment flows as edges, 165 features per node, 49 time steps. Labels: 4,545 illicit, 42,019 licit, the rest unknown.

The three files are about 450 MB unpacked and take a few minutes to download the first time. The join takes under a minute.

The loader joins the three files: features (no header in the raw file), classes and the edge list. `label` is 1 illicit, 0 licit, -1 unknown. Unknown rows stay in both tables because neighbour features need every node; the metric uses labelled rows only.

Before you change anything, save your own copy with File → Save a copy in
GitHub and give it a path with your name in it, such as
`notebooks/track09_<yourname>.ipynb`. The original stays clean so a
teammate can start from it.

## 1. Setup

Edit `REPO_URL` to your team's repository, then run the cell. It clones the
repository into this Colab session (or pulls, if it's already here), moves
into it and installs the two or three packages Colab doesn't ship with.

Tokens come from **Colab Secrets** (the key icon in the left sidebar), never
from a cell. The names it looks for are `GITHUB_TOKEN`, `KAGGLE_USERNAME` and
`KAGGLE_KEY`. If one is missing, the error says which. The getting-started
notebook shows how to create them.

In [ ]:
REPO_URL = "https://github.com/<your-team>/<your-repo>.git"   # <- edit this line

import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if "<your-team>" in REPO_URL:
        raise ValueError("Edit REPO_URL at the top of this cell to your team's repository first.")
    from google.colab import userdata

    def _secret(name):
        try:
            return userdata.get(name)
        except Exception as e:
            raise RuntimeError(
                f"Colab Secret {name} is missing or not shared with this notebook. "
                "Open the key icon in the left sidebar, add it, and switch on Notebook access."
            ) from e

    token = _secret("GITHUB_TOKEN")
    for name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        os.environ[name] = _secret(name)

    repo = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.isdir(repo):
        auth_url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", "--quiet", auth_url], check=True)
    os.chdir(repo)
    subprocess.run(["git", "pull", "--quiet"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # Not in Colab: assume this notebook is open from inside the repository.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

sys.path.insert(0, os.getcwd())
import config
print("repository:", os.getcwd(), "| template version:", config.TEMPLATE_VERSION)
print("TRACK in config.py:", config.TRACK, "" if config.TRACK else "<- one teammate sets this and pushes config.py")

## 2. Download and load

`load_track09()` returns the train pool (time steps 1 to 34), the test set (35 to 49) and the edge list.

In [ ]:
from src import data

pool, test, edges = data.load_track09()

Optional: cache a copy in Google Drive. Colab's disk is wiped between
sessions, so the download runs again every time. If that's slow for your
track, mount Drive and point `config.DATA_DIR` at a folder there before
loading. Everything below works the same either way.

In [ ]:
USE_DRIVE = False   # set True to keep the data between Colab sessions

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    from pathlib import Path
    drive.mount("/content/drive")
    config.DATA_DIR = Path("/content/drive/MyDrive/mais202_data")
    config.DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("data will be cached in", config.DATA_DIR)

## 3. Quick look

Numbers only; plots are D1.3.

In [ ]:
data.quick_look(pool, "label", "train pool")
data.quick_look(test, "label", "test")
print("edges:", f"{len(edges):,}")
print(pool.groupby("time_step")["label"].apply(lambda s: f"{int((s == 1).sum())} illicit / {int((s != -1).sum())} labelled").head(5))

## 4. The fixed test split

The loader returned two tables: a train pool and a test set. **The test set
is not to be touched until Deliverable 3.** Not for a plot, not for checking
a feature, not for picking a threshold. It's fixed in `src/data.py` so that
your D1, D2 and D3 numbers are measured on the same thing, and it only
measures anything if nobody has looked at it.

Your validation set comes out of the train pool, and that's the next cell.

The metric is F1 over all test steps and also per step. Steps differ a lot
in size and in how much is labelled, so report both; a single number hides
a step where the model falls apart. The same goes for your validation set:
the last few training steps are the natural choice, and look at them one by
one.

In [ ]:
print("train pool steps:", int(pool["time_step"].min()), "->", int(pool["time_step"].max()))
print("test steps:      ", int(test["time_step"].min()), "->", int(test["time_step"].max()))

## 5. Your validation split (TODO D1.2)

Carve a validation set out of `pool`, in `src/split.py`, as the function
`make_validation_split(pool)`. It lives in `src/` and not in this notebook
so that every teammate, every notebook and every model uses the same split;
three slightly different splits in three notebooks is how a team ends up
arguing about whose number is right.

For tracks with a time column, the validation set is the latest part of
the training period, so the model is judged on data from after what it
learned from. A random split on time-ordered data gives a validation score
that looks great and a test score that collapses. Tracks without time (6)
split at random, stratified on the label.

`split.check_split()` runs the two checks the deliverable asks for (no
identifier in both parts; training ends before validation begins), and
`pytest tests/test_leakage.py` runs them on your split. Green there is
D1.2 done.

In [ ]:
from src import split

# TODO D1.2: write make_validation_split() in src/split.py, then run this cell.
try:
    train, valid = split.make_validation_split(pool)
except NotImplementedError as e:
    train = valid = None
    print(e)
else:
    split.check_split(train, valid, *split.default_columns())
    print(f"train: {len(train):,} rows, valid: {len(valid):,} rows; both checks passed")

## 6. Features (TODO D1.4)

Features live in `src/features.py`, one function each, so that the
notebook, the tests and `predict()` all build them the same way. The file
has one worked example to show the shape. It's an example of a feature
function, not a feature you should rely on. Write at least three of your
own there, then call `build_features` here to check they behave.

In [ ]:
from src import features

# TODO D1.4: write your features in src/features.py, add them to build_features(), then run this.
features.build_features(pool.head(10))

## 7. The trivial baseline (TODO D1.5)

The baseline calls every row legitimate: every score is 0. It's here to be
scored with your metric and with accuracy, side by side, before you train
anything. On this track it will look very accurate and catch nothing. That
gap is the point of the evaluation metric section of your proposal.

Write your track's metric in `src/metrics.py` until
`pytest tests/test_metrics.py` is green. Then uncomment the lines below and
run them on **your validation set**.

In [ ]:
from src.models import always_legitimate
from sklearn.metrics import accuracy_score

baseline_scores = always_legitimate(len(pool))
print("baseline scores:", baseline_scores[:5], "... (all zeros)")
labelled = pool["label"] != -1          # only matters on Track 9, where -1 means unknown
print("accuracy on the train pool if we predict 'legitimate' everywhere:",
      round(accuracy_score(pool["label"][labelled], (baseline_scores[labelled] >= 0.5).astype(int)), 4))

# TODO D1.5: once `pytest tests/test_metrics.py` is green, score the baseline on YOUR validation set:
# from src import metrics
# y_valid = valid["label"].to_numpy()
# s_valid = always_legitimate(len(valid))
# print("headline metric:", metrics.f1_illicit(y_valid, (s_valid >= 0.5).astype(int)))
# print("accuracy:       ", accuracy_score(y_valid, (s_valid >= 0.5).astype(int)))

## 8. Neighbours

What makes this track a network problem: a transaction's neighbours on the
graph carry signal about it. `edges` has one row per payment flow, `txId1`
to `txId2`, and both ends are in the same time step. Features like "how
many of my neighbours are labelled illicit in training" or "the mean of my
neighbours' feature 3" are built from a join of `edges` with the node table.
Only training-period labels may be used in such features; the cell shows
the degree distribution to start from.

In [ ]:
degree = edges["txId1"].value_counts().rename("out") .to_frame().join(edges["txId2"].value_counts().rename("in"), how="outer").fillna(0)
print(degree.describe().round(2))

## 9. Running the tests

The metric tests fail until D1.5 is done; that's expected, and the message
says which function to write. The leakage tests on the provided split should
pass now. `test_predict.py` skips until D2.6.

In [ ]:
!python -m pytest -q tests/test_metrics.py tests/test_leakage.py

## Where to go next

In this order, because each step uses the one before:

1. `make_validation_split` in `src/split.py` until `pytest tests/test_leakage.py` is green (D1.2).
2. Two exploration plots, in your own notebook (D1.3).
3. Three features in `src/features.py` (D1.4).
4. The metric in `src/metrics.py` until the tests pass, then the baseline
   scored on validation with accuracy beside it (D1.5).
5. The cost table in `src/decision.py` (D1.6).
6. Write the report under the headings in `reports/Project Proposal.md`.

Work in a copy of this notebook with your name in the filename. Push code
in `src/` with git from a cell; save notebooks with File → Save a copy in
GitHub.